In [ ]:
%%bash
# Colab 2026.07，Linux x86_64。獨立環境，保留筆記本核心。
set -euo pipefail
cd /content
mkdir -p book-tools
curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C book-tools bin/micromamba
book-tools/bin/micromamba create -y -p /content/book-xtb -c conda-forge xtb

## 重繪書圖
取自 `medium_old/github/pyscf-xtb-benchmark-main/{xtb,pyscf}/results/` 的四對 JSON，檔名為分子鍵名加上 `_xtb_results.json` 或 `_pyscf_results.json`，欄位均為 `homo_lumo_gap_ev`。左列 GFN2-xTB、右列 M06-2X/6-31+G(d,p)，單位 eV。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# 支援在專案根目錄、book 子目錄或獨立 Colab 工作階段執行。
here = Path.cwd()
book_root = next((p for p in (here, *here.parents) if (p / "outline.yml").is_file()), None)
if book_root is None and (here / "book" / "outline.yml").is_file():
    book_root = here / "book"
fig_dir = (book_root or here) / "figures"
fig_dir.mkdir(exist_ok=True)
pairs = {'PrK2': (1.73570685, 4.91877091060635), 'p_nitroaniline': (2.57100406, 6.503989762873001), 'naphthalene': (3.24999464, 6.8814355393419735), 'caffeine': (3.42378812, 7.321045128993517)}
names = list(pairs)
xtb_gap, dft_gap = np.array(list(pairs.values())).T
print("DFT minus xTB (eV):", dft_gap - xtb_gap)

In [ ]:
fig, ax = plt.subplots(figsize=(7.1, 3.6))
y = np.arange(len(names))
ax.hlines(y, xtb_gap, dft_gap, colors="#a6b5bd", linewidth=2)
ax.scatter(xtb_gap, y, color="#996228", s=68, label="GFN2-xTB")
ax.scatter(dft_gap, y, color="#176d78", marker="s", s=56, label="M06-2X")
ax.set(yticks=y, yticklabels=["PrK2", "p-Nitroaniline", "Naphthalene", "Caffeine"],
       xlabel="Orbital gap (eV)", xlim=(0.9, 8))
ax.invert_yaxis()
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.18), ncol=2, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(fig_dir / "05-06-gfn2-xtb.png", dpi=160, bbox_inches="tight")
plt.show()

## 新計算（Colab 2026.07）
執行後會產生本次的獨立結果。原資料及書圖保留在上方。

In [ ]:
import json
import subprocess
import tempfile

xtb = ["/content/book-tools/bin/micromamba", "run", "-p", "/content/book-xtb", "xtb"]
subprocess.run([*xtb, "--version"], check=True)
run_dir = Path(tempfile.mkdtemp(prefix="ethyne-"))
(run_dir / "ethyne.xyz").write_text("4\nethyne\nH 0 0 0\nC 0 0 1\nC 0 0 2\nH 0 0 3\n")
result = subprocess.run(
    [*xtb, "ethyne.xyz", "--gfn", "2", "--opt"],
    cwd=run_dir, capture_output=True, text=True, check=True,
)
if "GEOMETRY OPTIMIZATION CONVERGED" not in result.stdout:
    raise RuntimeError("乙炔幾何尚未收斂，請查看計算紀錄")
subprocess.run([*xtb, "xtbopt.xyz", "--gfn", "2", "--json"], cwd=run_dir, check=True)
values = json.loads((run_dir / "xtbout.json").read_text())
print(f"Optimized ethyne gap: {values['HOMO-LUMO gap / eV']:.4f} eV")
print(run_dir)